<a href="https://colab.research.google.com/github/srikanth-ink/Movie-Recommendation-System/blob/main/personalized_recommendations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install keybert==0.9.0
!pip install sentence-transformers transformers --quiet
!pip install symspellpy
!pip install textblob
!python -m textblob.download_corpora

import os
import re
import random

import nltk
import numpy as np
import pandas as pd

from collections import Counter, defaultdict

from keybert import KeyBERT
from nltk.stem import WordNetLemmatizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from textblob import TextBlob

from google.colab import drive
drive.mount('/content/drive')

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package conll2000 to /root/nltk_data...
[nltk_data]   Package conll2000 is already up-to-date!
[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!
Finished.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# Load data
dta = pd.read_csv('/content/drive/MyDrive/RecommendMovies/reviews.csv', encoding='latin-1')
dta = dta.loc[:, ~dta.columns.str.contains('^Unnamed')]

# Display
dta.head()

,user_id,user_name,movie_id,movie_name,movie_rating,movie_review
0,1,roshan,101,american beauty,8.0,great movie! the story is new and the CG is up...
1,2,chethan,103,sitaramam,8.5,I was not intrested in watching the movie firs...
2,1,roshan,106,rrr,9.0,"the name ""ss rajamouli"" is enough to watch the..."
3,1,roshan,107,django unchained,5.0,pacing of the movie is too slow. at many parts...
4,1,roshan,110,premalu,8.0,enjoyed every scene of the movie. it was funny...


In [13]:
# Initialize KeyBERT with best model for English
kw_model = KeyBERT(model=SentenceTransformer('all-mpnet-base-v2'))

# Cluster definitions (unchanged)
clusters = {
    "Action": ["action", "action-packed", "fast-paced", "epic", "chase scenes", "fight sequences", "stunts", "must-watch for action lovers", "superhero", "face off", "adventure", "action thriller"],
    "Romance": ["emotional", "heartfelt", "love-story", "beautiful love", "chemistry between", "romance", "heartwarming", "chemistry"],
    "Comedy": ["comedy", "hilarious", "funny", "laughter", "humorous", "jokes", "entertaining", "entertained", "laugh"],
    "Horror": ["terrifying", "scares", "scary", "jump scares", "haunted", "fear", "tension", "creepy", "ghost", "killer", "suspense", "suspenseful", "thriller", "horror", "killing", "kill"],
    "Thriller": ["suspense", "killer", "kill", "tension", "seat-edge", "psychological", "trauma", "killings"],
    "General": []
}

# Create folder structure (your original code)
base_path = "/content/drive/MyDrive/RecommendMovies/clusters"
for cluster_name in clusters:
    os.makedirs(os.path.join(base_path, cluster_name, "movies"), exist_ok=True)
    os.makedirs(os.path.join(base_path, cluster_name, "users"), exist_ok=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [14]:
# Movie Clustering Thresholds
MIN_KEYWORD_MATCHES = 2
MIN_SENTENCE_LENGTH = 8
SENTIMENT_CUTOFF = -0.5
KEYWORD_CONFIDENCE = 0.25
MAX_CLUSTERS_PER_MOVIE = 2

# User Clustering Thresholds
MIN_USER_REVIEWS = 5
PRIMARY_THRESHOLD = 3
SECONDARY_THRESHOLD = 2
SECONDARY_RATIO = 0.5

# Confirmation of updated thresholds
print(f"Updated Thresholds - Movie Clustering: MIN_KEYWORD_MATCHES={MIN_KEYWORD_MATCHES}, MAX_CLUSTERS_PER_MOVIE={MAX_CLUSTERS_PER_MOVIE}")
print(f"Updated Thresholds - User Clustering: MIN_USER_REVIEWS={MIN_USER_REVIEWS}, PRIMARY_THRESHOLD={PRIMARY_THRESHOLD}, SECONDARY_THRESHOLD={SECONDARY_THRESHOLD}")


Updated Thresholds - Movie Clustering: MIN_KEYWORD_MATCHES=2, MAX_CLUSTERS_PER_MOVIE=2
Updated Thresholds - User Clustering: MIN_USER_REVIEWS=5, PRIMARY_THRESHOLD=3, SECONDARY_THRESHOLD=2


In [15]:
nltk.download(['punkt', 'wordnet'], quiet=True)
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    words = nltk.word_tokenize(text)
    return ' '.join([lemmatizer.lemmatize(word) for word in words])

def extract_keywords(text):
    sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?|\!)\s', text)
    all_keywords = []
    for sentence in sentences:
        if len(sentence.strip()) < MIN_SENTENCE_LENGTH:
            continue
        if TextBlob(sentence).sentiment.polarity < SENTIMENT_CUTOFF:
            continue
        processed = lemmatize_text(sentence.lower())
        keywords = kw_model.extract_keywords(
            processed,
            keyphrase_ngram_range=(1, 2),
            stop_words='english',
            top_n=6,
            diversity=0.3
        )
        all_keywords.extend([kw[0] for kw in keywords if kw[1] > KEYWORD_CONFIDENCE])
    return Counter(all_keywords)

cluster_assignments = defaultdict(list)
general_movies = set()

for _, row in dta.iterrows():
    movie_id, movie_name, review = row['movie_id'], row['movie_name'], row['movie_review']
    keyword_counts = extract_keywords(review)
    qualified_clusters = []
    for cluster, cluster_keywords in clusters.items():
        if cluster == "General":
            continue
        score = sum(
            count for kw, count in keyword_counts.items()
            if any(cluster_kw.lower()[:3] in kw.lower() for cluster_kw in cluster_keywords)
        )
        if score >= MIN_KEYWORD_MATCHES:
            qualified_clusters.append((cluster, score))
    if qualified_clusters:
        qualified_clusters.sort(key=lambda x: -x[1])
        top_score = qualified_clusters[0][1]
        for i, (cluster, score) in enumerate(qualified_clusters[:MAX_CLUSTERS_PER_MOVIE]):
            weight = min(100, int(70 + 30 * (score / top_score)))
            cluster_assignments[cluster].append((movie_id, movie_name, weight))
    else:
        cluster_assignments["General"].append((movie_id, movie_name, 100))
        general_movies.add(movie_id)

cluster_assignments["General"] = [
    m for m in cluster_assignments["General"]
    if m[0] not in set(
        mid for cluster in clusters
        for mid, _, _ in cluster_assignments.get(cluster, [])
        if cluster != "General"
    )
]

for cluster in clusters:
    os.makedirs(os.path.join(base_path, cluster, "movies"), exist_ok=True)
    with open(os.path.join(base_path, cluster, "movies", "assigned_movies.txt"), 'w') as f:
        for movie in cluster_assignments.get(cluster, []):
            f.write(f"{movie[0]},{movie[1]},{movie[2]}\n")

print("\nOptimized Cluster Distribution:")
print("="*50)
for cluster in sorted(clusters.keys()):
    movies = cluster_assignments.get(cluster, [])
    print(f"{cluster:>8}: {len(movies):>3} movies")


Optimized Cluster Distribution:
  Action:  40 movies
  Comedy:  31 movies
 General:   0 movies
  Horror:  72 movies
 Romance:  27 movies
Thriller:  10 movies


In [16]:
from collections import defaultdict

def assign_users_to_clusters():
    user_reviews = dta[dta['movie_rating'] >= 6]
    user_review_counts = user_reviews['user_id'].value_counts()
    qualified_users = user_review_counts[user_review_counts >= MIN_USER_REVIEWS].index
    user_reviews = user_reviews[user_reviews['user_id'].isin(qualified_users)]

    user_reviews = user_reviews.groupby('user_id').agg({
        'user_name': 'first',
        'movie_review': ' '.join
    }).reset_index()

    user_assignments = defaultdict(list)
    general_count = 0

    for _, row in user_reviews.iterrows():
        user_id, user_name, reviews = row['user_id'], row['user_name'], row['movie_review']
        keyword_counts = extract_keywords(reviews)
        cluster_scores = {}
        for cluster, cluster_keywords in clusters.items():
            if cluster == "General":
                continue
            score = sum(
                count for kw, count in keyword_counts.items()
                if any(cluster_kw.lower()[:3] in kw.lower() for cluster_kw in cluster_keywords)
            )
            if score > 0:
                cluster_scores[cluster] = score

        assigned = []
        if cluster_scores:
            top_clusters = sorted(cluster_scores.items(), key=lambda x: -x[1])[:2]
            if top_clusters[0][1] >= PRIMARY_THRESHOLD:
                assigned.append((top_clusters[0][0], 100))
                if len(top_clusters) > 1 and top_clusters[1][1] >= SECONDARY_THRESHOLD:
                    if top_clusters[1][1] / top_clusters[0][1] > SECONDARY_RATIO:
                        assigned.append((top_clusters[1][0], int(100 * top_clusters[1][1]/top_clusters[0][1])))

        if not assigned:
            assigned.append(("General", 100))
            general_count += 1

        for cluster, weight in assigned:
            user_assignments[cluster].append((user_id, user_name, weight))

    for cluster, users in user_assignments.items():
        users_dir = os.path.join(base_path, cluster, "users")
        os.makedirs(users_dir, exist_ok=True)
        with open(os.path.join(users_dir, "assigned_users.txt"), 'w') as f:
            for user_id, user_name, weightage in users:
                f.write(f"{user_id},{user_name},{weightage:.2f}\n")

    print(f"\nUser clustering complete. Assigned to General: {general_count}/{len(user_reviews)}")
    print("\nUser cluster distribution:")
    for cluster in sorted(clusters.keys()):
        users = user_assignments.get(cluster, [])
        print(f"{cluster:>8}: {len(users):>2} users")

assign_users_to_clusters()



User clustering complete. Assigned to General: 0/13

User cluster distribution:
  Action:  3 users
  Comedy:  4 users
 General:  0 users
  Horror: 10 users
 Romance:  5 users
Thriller:  0 users


In [17]:
# === Load user weightages ===
user_weightages = defaultdict(dict)
for cluster in clusters:
    users_file = os.path.join(base_path, cluster, "users", "assigned_users.txt")
    if os.path.exists(users_file):
        with open(users_file, 'r') as f:
            for line in f:
                parts = line.strip().split(',')
                if len(parts) == 3: # Ensure we have exactly 3 parts before unpacking
                    user_id, user_name, weight = parts
                    user_weightages[int(user_id)][cluster] = float(weight)
                else:
                    print(f"Skipping malformed line in {users_file}: {line.strip()}")

# === Load movies by cluster ===
cluster_movies = defaultdict(list)
for cluster in clusters:
    movies_file = os.path.join(base_path, cluster, "movies", "assigned_movies.txt")
    if os.path.exists(movies_file):
        with open(movies_file, 'r') as f:
            for line in f:
                parts = line.strip().split(',')
                if len(parts) == 3: # Ensure we have exactly 3 parts
                    movie_id, movie_name, weight = parts
                    cluster_movies[cluster].append((int(movie_id), movie_name, int(weight)))
                else:
                    print(f"Skipping malformed line in {movies_file}: {line.strip()}")

# === Fallback for popular movies ===
def get_most_popular_movies(exclude_set, count):
    popular = (
        dta[~dta['movie_id'].isin(exclude_set)]
        .groupby(['movie_id', 'movie_name'])['movie_rating']
        .mean()
        .sort_values(ascending=False)
        .head(count)
        .reset_index()
    )
    return list(popular['movie_name'])

# === Recommendation function ===
def recommend_movies(target_user_id):
    TARGET_RECOMMENDATIONS = 5
    TARGET_PRIMARY = 3
    TARGET_SECONDARY = 1
    TARGET_DIVERSITY = 1

    watched_movies = set(dta[dta['user_id'] == target_user_id]['movie_id'])
    recommendations = []
    recommended_movie_ids = set()
    diversity_candidates = []

    # Get all user-movie rating vectors (sparse)
    user_movie_matrix = dta.pivot_table(index='user_id', columns='movie_id', values='movie_rating', fill_value=0)

    def get_available_movies(cluster_name):
        return [m for m in cluster_movies.get(cluster_name, [])
                if m[0] not in watched_movies and m[0] not in recommended_movie_ids]

    def get_most_similar_users(user_id, cluster):
        cluster_users = [uid for uid in user_weightages if cluster in user_weightages[uid] and uid != user_id]
        if not cluster_users or user_id not in user_movie_matrix.index:
            return []
        target_vector = user_movie_matrix.loc[[user_id]]
        sim_users = []
        for uid in cluster_users:
            if uid in user_movie_matrix.index:
                score = cosine_similarity(target_vector, user_movie_matrix.loc[[uid]])[0][0]
                sim_users.append((uid, score))
        sim_users.sort(key=lambda x: -x[1])  # Descending similarity
        return sim_users

    def movie_cluster_membership(movie_id):
        for cluster, movies in cluster_movies.items():
            if any(mid == movie_id for mid, _, _ in movies):
                return cluster
        return None

    def recommend_from_cluster(cluster_name, user_id, count, diversity_mode=False):
        available_movies = get_available_movies(cluster_name)
        recommended_movies = []

        if diversity_mode:
            random.shuffle(diversity_candidates)
            for movie_name in diversity_candidates:
                if movie_name not in recommended_movies:
                    recommended_movies.append((movie_name, "Diversity"))
                    if len(recommended_movies) == count:
                        break
            return recommended_movies

        similar_users = get_most_similar_users(user_id, cluster_name)

        for sim_user, _ in similar_users:
            sim_user_ratings = dta[dta['user_id'] == sim_user]
            highly_rated = sim_user_ratings[sim_user_ratings['movie_rating'] > 7]

            for _, row in highly_rated.iterrows():
                movie_id = row['movie_id']
                movie_name = row['movie_name']
                if movie_id in watched_movies or movie_id in recommended_movie_ids:
                    continue
                cluster = movie_cluster_membership(movie_id)
                if cluster == cluster_name:
                    recommended_movies.append((movie_name, cluster_name))
                    recommended_movie_ids.add(movie_id)
                    if len(recommended_movies) == count:
                        return recommended_movies
                elif cluster != primary_cluster and cluster != secondary_cluster:
                    diversity_candidates.append(movie_name)

        return recommended_movies

    primary_cluster = None
    secondary_cluster = None
    if target_user_id in user_weightages and user_weightages[target_user_id]:
        sorted_clusters = sorted(user_weightages[target_user_id].items(), key=lambda x: -x[1])
        primary_cluster = sorted_clusters[0][0]
        secondary_cluster = sorted_clusters[1][0] if len(sorted_clusters) > 1 else None

    def update_and_track(recs, target_count):
        shortfall = target_count - len(recs)
        recommendations.extend(recs)
        return max(0, shortfall)

    primary_recs = recommend_from_cluster(primary_cluster, target_user_id, TARGET_PRIMARY)
    gap = update_and_track(primary_recs, TARGET_PRIMARY)

    total_secondary_needed = TARGET_SECONDARY + gap
    secondary_recs = recommend_from_cluster(secondary_cluster, target_user_id, total_secondary_needed)
    gap = update_and_track(secondary_recs, TARGET_SECONDARY + gap)

    diversity_recs = recommend_from_cluster("General", target_user_id, TARGET_DIVERSITY + gap, diversity_mode=True)
    gap = update_and_track(diversity_recs, TARGET_DIVERSITY + gap)

    if gap > 0:
        popular_movies = get_most_popular_movies(watched_movies.union(recommended_movie_ids), gap)
        for movie in popular_movies:
            recommendations.append((movie, "Popular"))

    return recommendations[:TARGET_RECOMMENDATIONS]

# Example usage:
user_id = input("Enter user ID for recommendation: ")
print(recommend_movies(int(user_id)))

Skipping malformed line in /content/drive/MyDrive/RecommendMovies/clusters/General/users/assigned_users.txt: baba
Skipping malformed line in /content/drive/MyDrive/RecommendMovies/clusters/General/users/assigned_users.txt: chethan
Skipping malformed line in /content/drive/MyDrive/RecommendMovies/clusters/General/users/assigned_users.txt: kamalnath
Skipping malformed line in /content/drive/MyDrive/RecommendMovies/clusters/General/users/assigned_users.txt: kavya
Skipping malformed line in /content/drive/MyDrive/RecommendMovies/clusters/General/users/assigned_users.txt: mahesh
Skipping malformed line in /content/drive/MyDrive/RecommendMovies/clusters/General/users/assigned_users.txt: roshan
Skipping malformed line in /content/drive/MyDrive/RecommendMovies/clusters/General/users/assigned_users.txt: saketh
Skipping malformed line in /content/drive/MyDrive/RecommendMovies/clusters/General/users/assigned_users.txt: sravan
Skipping malformed line in /content/drive/MyDrive/RecommendMovies/clust